# Crop Disease Forecast — Preprocessing
**Mini Project | DS/AIML-A 2026**

This notebook demonstrates the full data cleaning and preprocessing pipeline step by step.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
print('Libraries loaded!')

## 2. Load Raw Data

In [ ]:
df = pd.read_csv('../dataset/raw_data/climate_data.csv', parse_dates=['date'])
print('Raw data shape:', df.shape)
df.head()

## 3. Check Missing Values

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)

# Visualise missing values
plt.figure(figsize=(10, 4))
missing.plot(kind='bar', color='#e74c3c')
plt.title('Missing Values Per Column')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/graphs/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Handle Missing Values

In [ ]:
# Fill missing values with column mean
df.fillna(df.mean(numeric_only=True), inplace=True)
print('Missing values after filling:')
print(df.isnull().sum())

## 5. Remove Outliers

In [ ]:
# Show boxplots before removing outliers
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df['temp_max'].plot(kind='box', ax=axes[0], title='Max Temperature')
df['humidity_max'].plot(kind='box', ax=axes[1], title='Max Humidity')
df['rainfall_mm'].plot(kind='box', ax=axes[2], title='Rainfall')

plt.tight_layout()
plt.savefig('../outputs/graphs/boxplots_before.png', dpi=150, bbox_inches='tight')
plt.show()

# Remove temperature outliers
before = len(df)
df = df[(df['temp_max'] >= 10) & (df['temp_max'] <= 50)]
df = df[(df['temp_min'] >= 10) & (df['temp_min'] <= 50)]
print(f'Rows removed: {before - len(df)}')
print(f'Rows remaining: {len(df)}')

## 6. Feature Engineering

In [ ]:
# Add new useful columns
df['month']         = df['date'].dt.month
df['year']          = df['date'].dt.year
df['temp_range']    = df['temp_max'] - df['temp_min']
df['high_humidity'] = (df['humidity_avg'] > 80).astype(int)
df['heavy_rain']    = (df['rainfall_mm'] > 10).astype(int)

def get_season(month):
    if month in [6,7,8,9]:    return 'Monsoon'
    elif month in [10,11]:     return 'Post-Monsoon'
    elif month in [12,1,2]:    return 'Winter'
    else:                      return 'Summer'

df['season'] = df['month'].apply(get_season)

print('New columns added!')
df[['date','temp_range','high_humidity','heavy_rain','season']].head(10)

## 7. Season Distribution

In [ ]:
plt.figure(figsize=(8, 4))
season_colors = {'Monsoon':'#3498db','Post-Monsoon':'#e67e22','Winter':'#9b59b6','Summer':'#e74c3c'}
counts = df['season'].value_counts()
counts.plot(kind='bar', color=[season_colors[s] for s in counts.index])
plt.title('Number of Days Per Season (2020-2023)')
plt.ylabel('Days')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/graphs/season_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Cleaned Data

In [ ]:
df.to_csv('../dataset/processed_data/climate_cleaned.csv', index=False)
print('Cleaned data saved!')
print('Final shape:', df.shape)
df.head()